# NCHU Rental Model — Colab T4 Training

**使用前請確認**：Runtime → Change runtime type → **T4 GPU**

流程：
1. Clone repo
2. 安裝依賴
3. 設定超參數
4. 確認訓練資料
5. 訓練 RoBERTa cross-encoder
6. 下載量化 ONNX 模型

## Cell 0 — 確認 GPU

In [ ]:
import torch
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
    print('VRAM:', round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1), 'GB')
else:
    raise RuntimeError('⚠️  沒有 GPU！請到 Runtime → Change runtime type → T4 GPU 再重跑')

## Cell 1 — Clone Repo

In [ ]:
import os

REPO_URL = "https://github.com/eric20041027/Renting-recommendation-ONNX.git"
REPO_DIR = "/content/Renting-recommendation-ONNX"

if os.path.exists(REPO_DIR):
    print('Repo 已存在，更新至最新...')
    %cd {REPO_DIR}
    !git pull
else:
    !git clone {REPO_URL} {REPO_DIR}
    %cd {REPO_DIR}

!git log --oneline -3

## Cell 2 — 安裝依賴

In [ ]:
!pip install -q \
    'transformers>=4.35' \
    datasets \
    'onnx>=1.14' \
    'onnxruntime>=1.17' \
    'pandas>=2.0' \
    'pydantic>=2.0' \
    accelerate \
    onnxscript

print('✅ 安裝完成')

## Cell 3 — 設定訓練超參數

In [ ]:
import os

# 模型儲存路徑
os.environ["SAVED_MODEL_DIR"]      = "/content/saved_models/rbt6_finetuned"
os.environ["ONNX_OUTPUT_PATH"]     = "/content/saved_models/my_custom_model.onnx"
os.environ["QUANTIZED_MODEL_PATH"] = "/content/saved_models/my_custom_model_quant.onnx"

# T4 最佳化超參數
os.environ["BATCH_SIZE"]    = "64"
os.environ["NUM_EPOCHS"]    = "10"
os.environ["LEARNING_RATE"] = "2e-5"
os.environ["MAX_LENGTH"]    = "64"
os.environ["WARMUP_STEPS"]  = "500"
os.environ["EARLY_STOPPING_PATIENCE"] = "3"
os.environ["RANDOM_SEED"]   = "42"

# opset 18：新版 torch.onnx 最低支援 18，勿改為 15
os.environ["ONNX_OPSET_VERSION"] = "18"

# FP16 關閉：FGMTrainer 對抗訓練步驟與 fp16 grad scaler 不相容
os.environ["FP16"] = "false"

os.environ["ENABLE_ADVERSARIAL_TRAINING"] = "true"
os.environ["ENABLE_HARD_MINING"]          = "true"
os.environ["ENABLE_QUANTIZATION"]         = "true"

os.makedirs("/content/saved_models", exist_ok=True)
print('✅ 超參數設定完成')
print(f'   BATCH_SIZE={os.environ["BATCH_SIZE"]}  NUM_EPOCHS={os.environ["NUM_EPOCHS"]}  ONNX_OPSET={os.environ["ONNX_OPSET_VERSION"]}')

## Cell 4 — 確認訓練資料

In [ ]:
import json, os
from collections import Counter

train_path = 'data/processed/recommendation_train.json'

if not os.path.exists(train_path):
    print('⏳ 訓練資料不存在，重新生成...')
    !python3 pipeline/data_prep/generate_dataset.py
    print('✅ 生成完成')

for split in ['train', 'dev', 'test']:
    with open(f'data/processed/recommendation_{split}.json') as f:
        data = json.load(f)
    pos = sum(1 for s in data if s['label'] == 1)
    neg = len(data) - pos
    ct  = sum(1 for s in data if s.get('conflict_type'))
    print(f'{split:5s}: {len(data):6d} 筆  pos={pos} neg={neg} ratio=1:{neg/pos:.1f}  conflict_type={ct}')

print()
print('conflict_type 分布 (train):')
with open('data/processed/recommendation_train.json') as f:
    train = json.load(f)
for k, v in Counter(s.get('conflict_type') for s in train if s.get('conflict_type')).most_common():
    print(f'  {k}: {v}')

## Cell 5 — 開始訓練

T4 預計時間：**20–40 分鐘**（10 epochs，37k 樣本）

In [ ]:
# 跳過 Phase 1（爬蟲）和 Phase 2（資料準備），只執行 Phase 3（訓練）
!python3 pipeline_runner.py --skip-phase 1 --skip-phase 2

## Cell 6 — 確認訓練產出

In [ ]:
import os

def check(path, label):
    if os.path.exists(path):
        mb = os.path.getsize(path) / 1024 / 1024
        print(f'✅ {label}: {path} ({mb:.1f} MB)')
    else:
        print(f'❌ {label}: {path} 不存在')

saved_dir  = os.environ["SAVED_MODEL_DIR"]
onnx_path  = os.environ["ONNX_OUTPUT_PATH"]
quant_path = os.environ["QUANTIZED_MODEL_PATH"]

check(os.path.join(saved_dir, 'config.json'), 'PyTorch 模型')
check(onnx_path,  'ONNX 模型')
check(quant_path, '量化 ONNX 模型')

## Cell 7 — 下載量化 ONNX 模型

In [ ]:
from google.colab import files
import os

quant_path = os.environ["QUANTIZED_MODEL_PATH"]

if os.path.exists(quant_path):
    print(f'⬇️  下載：{quant_path}')
    files.download(quant_path)
else:
    print('❌ 量化模型不存在，請確認 Cell 5 訓練是否完成')

## Cell 8 — （可選）下載完整 PyTorch 模型

In [ ]:
import os, shutil
from google.colab import files

saved_dir = os.environ["SAVED_MODEL_DIR"]
zip_path  = '/content/rbt6_finetuned.zip'

!zip -r {zip_path} {saved_dir}
files.download(zip_path)

## Cell 9 — （可選）自動存到 Google Drive

In [ ]:
from google.colab import drive
import os, shutil

drive.mount('/content/drive')

dst = '/content/drive/MyDrive/renting_model'
os.makedirs(dst, exist_ok=True)

quant_path = os.environ["QUANTIZED_MODEL_PATH"]
saved_dir  = os.environ["SAVED_MODEL_DIR"]

if os.path.exists(quant_path):
    shutil.copy(quant_path, dst)
    print(f'✅ 量化模型 → {dst}')

if os.path.exists(saved_dir):
    shutil.copytree(saved_dir, os.path.join(dst, 'rbt6_finetuned'), dirs_exist_ok=True)
    print(f'✅ PyTorch 模型 → {dst}/rbt6_finetuned')